# Telco Customer Churn — lgbm_numpy Training Pipeline

This notebook walks through the full pipeline:
1. **Data Exploration** — shapes, types, distributions, missing values, churn rates
2. **Data Processing** — handle missing values, label-encode categoricals, standardize numerics
3. **Training** — train an `LGBMClassifier` with early stopping
4. **Cross-Validation** — k-fold CV to estimate generalisation
5. **Hyperparameter Tuning** — grid search over key params
6. **Final Model** — retrain with best params, evaluate, save

In [1]:
import sys, os

# Ensure lgbm_numpy is importable — find project root
_cwd = os.getcwd()
if os.path.isdir(os.path.join(_cwd, "lgbm_numpy")):
    PROJECT_ROOT = _cwd
elif os.path.isdir(os.path.join(_cwd, "..", "lgbm_numpy")):
    PROJECT_ROOT = os.path.abspath(os.path.join(_cwd, ".."))
elif os.path.isdir(os.path.join(_cwd, "..", "..", "lgbm_numpy")):
    PROJECT_ROOT = os.path.abspath(os.path.join(_cwd, "..", ".."))
else:
    raise FileNotFoundError("Cannot locate lgbm_numpy package")
sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import csv
import time
from collections import Counter
from itertools import product

from lgbm_numpy.api import LGBMClassifier
from lgbm_numpy.metrics import accuracy, logloss, auc, f1_score, precision, recall

np.random.seed(42)
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"lgbm_numpy found: {os.path.isdir(os.path.join(PROJECT_ROOT, 'lgbm_numpy'))}")
print("Imports OK")

PROJECT_ROOT = /home/ei/Projects/UNI/LightGBMLibrary
lgbm_numpy found: True
Imports OK


---
## 1. Data Loading

In [2]:
DATA_PATH = os.path.join(PROJECT_ROOT, "dataset.csv")

with open(DATA_PATH) as f:
    raw_rows = list(csv.DictReader(f))

columns = list(raw_rows[0].keys())
print(f"Loaded {len(raw_rows)} rows, {len(columns)} columns")
print(f"Columns: {columns}")

Loaded 7043 rows, 21 columns
Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


---
## 2. Data Exploration

In [3]:
# --- 2a. First rows ---
print("=== First 5 rows ===")
header = "  ".join(f"{c:>15s}" for c in columns)
print(header)
for r in raw_rows[:5]:
    print("  ".join(f"{r[c]:>15s}" for c in columns))

=== First 5 rows ===
     customerID           gender    SeniorCitizen          Partner       Dependents           tenure     PhoneService    MultipleLines  InternetService   OnlineSecurity     OnlineBackup  DeviceProtection      TechSupport      StreamingTV  StreamingMovies         Contract  PaperlessBilling    PaymentMethod   MonthlyCharges     TotalCharges            Churn
     7590-VHVEG           Female                0              Yes               No                1               No  No phone service              DSL               No              Yes               No               No               No               No   Month-to-month              Yes  Electronic check            29.85            29.85               No
     5575-GNVDE             Male                0               No               No               34              Yes               No              DSL              Yes               No              Yes               No               No               No         O

In [4]:
# --- 2b. Column types & missing values ---
print("=== Column Analysis ===")
numeric_cols = []
categorical_cols = []
target_col = "Churn"
drop_cols = ["customerID"]
total = len(raw_rows)

for col in columns:
    if col in drop_cols or col == target_col:
        continue
    blanks = sum(1 for row in raw_rows if row[col].strip() == "")
    unique_vals = set(row[col].strip() for row in raw_rows if row[col].strip() != "")
    is_numeric = False
    if blanks < total:
        try:
            for v in unique_vals:
                float(v)
            is_numeric = True
        except ValueError:
            pass
    if is_numeric:
        numeric_cols.append(col)
        vals = np.array([float(row[col].strip()) for row in raw_rows if row[col].strip() != ""])
        print(f"  {col:25s} NUMERIC   unique={len(unique_vals):5d}  blanks={blanks:4d}  "
              f"min={vals.min():10.2f}  max={vals.max():10.2f}  mean={vals.mean():.2f}")
    else:
        categorical_cols.append(col)
        top_vals = sorted(unique_vals)[:6]
        print(f"  {col:25s} CATEG     unique={len(unique_vals):5d}  blanks={blanks:4d}  vals={top_vals}")

print(f"\nNumeric features ({len(numeric_cols)}):       {numeric_cols}")
print(f"Categorical features ({len(categorical_cols)}):  {categorical_cols}")

=== Column Analysis ===
  gender                    CATEG     unique=    2  blanks=   0  vals=['Female', 'Male']
  SeniorCitizen             NUMERIC   unique=    2  blanks=   0  min=      0.00  max=      1.00  mean=0.16
  Partner                   CATEG     unique=    2  blanks=   0  vals=['No', 'Yes']
  Dependents                CATEG     unique=    2  blanks=   0  vals=['No', 'Yes']
  tenure                    NUMERIC   unique=   73  blanks=   0  min=      0.00  max=     72.00  mean=32.37
  PhoneService              CATEG     unique=    2  blanks=   0  vals=['No', 'Yes']
  MultipleLines             CATEG     unique=    3  blanks=   0  vals=['No', 'No phone service', 'Yes']
  InternetService           CATEG     unique=    3  blanks=   0  vals=['DSL', 'Fiber optic', 'No']
  OnlineSecurity            CATEG     unique=    3  blanks=   0  vals=['No', 'No internet service', 'Yes']
  OnlineBackup              CATEG     unique=    3  blanks=   0  vals=['No', 'No internet service', 'Yes']
  D

In [5]:
# --- 2c. Target distribution ---
target_counts = Counter(row[target_col] for row in raw_rows)
print("=== Target Distribution (Churn) ===")
for label, count in sorted(target_counts.items()):
    print(f"  {label:5s}: {count:5d} ({100*count/total:.1f}%)")
print(f"  Imbalance ratio: {max(target_counts.values())/min(target_counts.values()):.2f}x")

=== Target Distribution (Churn) ===
  No   :  5174 (73.5%)
  Yes  :  1869 (26.5%)
  Imbalance ratio: 2.77x


In [6]:
# --- 2d. Churn rate by categorical feature ---
print("=== Churn Rate by Categorical Feature ===")
for col in categorical_cols:
    print(f"\n  {col}:")
    groups = {}
    for row in raw_rows:
        val = row[col]
        if val not in groups:
            groups[val] = {"total": 0, "churned": 0}
        groups[val]["total"] += 1
        if row[target_col] == "Yes":
            groups[val]["churned"] += 1
    for val in sorted(groups.keys()):
        g = groups[val]
        rate = 100 * g["churned"] / g["total"] if g["total"] > 0 else 0
        print(f"    {val:35s}: {g['churned']:4d}/{g['total']:4d} = {rate:5.1f}%")

=== Churn Rate by Categorical Feature ===

  gender:
    Female                             :  939/3488 =  26.9%
    Male                               :  930/3555 =  26.2%

  Partner:
    No                                 : 1200/3641 =  33.0%
    Yes                                :  669/3402 =  19.7%

  Dependents:
    No                                 : 1543/4933 =  31.3%
    Yes                                :  326/2110 =  15.5%

  PhoneService:
    No                                 :  170/ 682 =  24.9%
    Yes                                : 1699/6361 =  26.7%

  MultipleLines:
    No                                 :  849/3390 =  25.0%
    No phone service                   :  170/ 682 =  24.9%
    Yes                                :  850/2971 =  28.6%

  InternetService:
    DSL                                :  459/2421 =  19.0%
    Fiber optic                        : 1297/3096 =  41.9%
    No                                 :  113/1526 =   7.4%

  OnlineSecurity:
    No

---
## 3. Data Processing

In [7]:
# --- 3a. Build label encoders for categoricals ---
label_encoders = {}  # col -> {string_val: int}
for col in categorical_cols:
    unique_sorted = sorted(set(row[col].strip() for row in raw_rows))
    label_encoders[col] = {v: i for i, v in enumerate(unique_sorted)}

feature_cols = [c for c in columns if c not in drop_cols + [target_col]]

print("Label Encoders:")
for col, enc in label_encoders.items():
    print(f"  {col:25s} -> {enc}")

Label Encoders:
  gender                    -> {'Female': 0, 'Male': 1}
  Partner                   -> {'No': 0, 'Yes': 1}
  Dependents                -> {'No': 0, 'Yes': 1}
  PhoneService              -> {'No': 0, 'Yes': 1}
  MultipleLines             -> {'No': 0, 'No phone service': 1, 'Yes': 2}
  InternetService           -> {'DSL': 0, 'Fiber optic': 1, 'No': 2}
  OnlineSecurity            -> {'No': 0, 'No internet service': 1, 'Yes': 2}
  OnlineBackup              -> {'No': 0, 'No internet service': 1, 'Yes': 2}
  DeviceProtection          -> {'No': 0, 'No internet service': 1, 'Yes': 2}
  TechSupport               -> {'No': 0, 'No internet service': 1, 'Yes': 2}
  StreamingTV               -> {'No': 0, 'No internet service': 1, 'Yes': 2}
  StreamingMovies           -> {'No': 0, 'No internet service': 1, 'Yes': 2}
  Contract                  -> {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
  PaperlessBilling          -> {'No': 0, 'Yes': 1}
  PaymentMethod             -> {'Ban

In [8]:
# --- 3b. Encode all rows into a numpy array ---
def encode_row(row):
    feats = []
    for col in feature_cols:
        val = row[col].strip()
        if col in numeric_cols:
            feats.append(float(val) if val != "" else 0.0)
        else:
            feats.append(float(label_encoders[col].get(val, 0)))
    return feats

X = np.array([encode_row(row) for row in raw_rows], dtype=np.float64)
y = np.array([1.0 if row[target_col] == "Yes" else 0.0 for row in raw_rows])

print(f"X shape: {X.shape}  y shape: {y.shape}")
print(f"Churn rate: {y.mean():.3f}")
print(f"\nSample features (row 0):")
for col, val in zip(feature_cols, X[0]):
    print(f"  {col:25s} = {val}")

X shape: (7043, 19)  y shape: (7043,)
Churn rate: 0.265

Sample features (row 0):
  gender                    = 0.0
  SeniorCitizen             = 0.0
  Partner                   = 1.0
  Dependents                = 0.0
  tenure                    = 1.0
  PhoneService              = 0.0
  MultipleLines             = 1.0
  InternetService           = 0.0
  OnlineSecurity            = 0.0
  OnlineBackup              = 2.0
  DeviceProtection          = 0.0
  TechSupport               = 0.0
  StreamingTV               = 0.0
  StreamingMovies           = 0.0
  Contract                  = 0.0
  PaperlessBilling          = 1.0
  PaymentMethod             = 2.0
  MonthlyCharges            = 29.85
  TotalCharges              = 29.85


In [9]:
# --- 3c. Standardize all features (zero mean, unit variance) ---
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
X_std[X_std == 0] = 1.0

X_scaled = (X - X_mean) / X_std

print("After standardization:")
print(f"  {'Feature':25s} {'Mean':>8s} {'Std':>8s}")
for i, col in enumerate(feature_cols):
    print(f"  {col:25s} {X_scaled[:, i].mean():8.4f} {X_scaled[:, i].std():8.4f}")

After standardization:
  Feature                       Mean      Std
  gender                     -0.0000   1.0000
  SeniorCitizen              -0.0000   1.0000
  Partner                     0.0000   1.0000
  Dependents                  0.0000   1.0000
  tenure                     -0.0000   1.0000
  PhoneService                0.0000   1.0000
  MultipleLines               0.0000   1.0000
  InternetService             0.0000   1.0000
  OnlineSecurity              0.0000   1.0000
  OnlineBackup                0.0000   1.0000
  DeviceProtection           -0.0000   1.0000
  TechSupport                -0.0000   1.0000
  StreamingTV                -0.0000   1.0000
  StreamingMovies             0.0000   1.0000
  Contract                   -0.0000   1.0000
  PaperlessBilling           -0.0000   1.0000
  PaymentMethod              -0.0000   1.0000
  MonthlyCharges             -0.0000   1.0000
  TotalCharges               -0.0000   1.0000


In [10]:
# --- 3d. Train / Test split (80/20, stratified) ---
pos_idx = np.where(y == 1)[0]
neg_idx = np.where(y == 0)[0]
np.random.shuffle(pos_idx)
np.random.shuffle(neg_idx)

n_pos_train = int(0.8 * len(pos_idx))
n_neg_train = int(0.8 * len(neg_idx))
train_idx = np.concatenate([pos_idx[:n_pos_train], neg_idx[:n_neg_train]])
test_idx = np.concatenate([pos_idx[n_pos_train:], neg_idx[n_neg_train:]])
np.random.shuffle(train_idx)
np.random.shuffle(test_idx)

X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Train: {len(y_train)} samples  (churn rate: {y_train.mean():.3f})")
print(f"Test:  {len(y_test)} samples   (churn rate: {y_test.mean():.3f})")

Train: 5634 samples  (churn rate: 0.265)
Test:  1409 samples   (churn rate: 0.265)


---
## 4. Model Training

In [11]:
# --- 4a. Baseline model ---
print("=== Training Baseline Model ===")
model = LGBMClassifier(
    objective="binary",
    n_estimators=150,
    learning_rate=0.1,
    num_leaves=31,
    min_data_in_leaf=20,
    lambda_l1=0.0,
    lambda_l2=1.0,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    max_bin=63,
    early_stopping_rounds=20,
    verbose=True,
    seed=42,
)

t0 = time.time()
model.fit(X_train, y_train, eval_set=[(X_test, y_test)])
print(f"\nTraining time: {time.time()-t0:.1f}s")

=== Training Baseline Model ===
[    0] train loss: 0.550162 | valid_0: 0.552547
[    1] train loss: 0.527186 | valid_0: 0.532055
[    2] train loss: 0.507825 | valid_0: 0.516419
[    3] train loss: 0.490793 | valid_0: 0.502296
[    4] train loss: 0.476752 | valid_0: 0.490686
[    5] train loss: 0.465541 | valid_0: 0.481849
[    6] train loss: 0.455814 | valid_0: 0.474751
[    7] train loss: 0.447373 | valid_0: 0.469070
[    8] train loss: 0.440198 | valid_0: 0.463142
[    9] train loss: 0.433837 | valid_0: 0.458240
[   10] train loss: 0.428067 | valid_0: 0.454232
[   11] train loss: 0.422503 | valid_0: 0.450237
[   12] train loss: 0.417515 | valid_0: 0.447458
[   13] train loss: 0.413758 | valid_0: 0.444471
[   14] train loss: 0.410413 | valid_0: 0.441896
[   15] train loss: 0.407277 | valid_0: 0.440193
[   16] train loss: 0.404131 | valid_0: 0.438520
[   17] train loss: 0.401283 | valid_0: 0.437493
[   18] train loss: 0.398603 | valid_0: 0.436162
[   19] train loss: 0.396162 | valid_

In [12]:
# --- 4b. Evaluate on test set ---
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("=== Baseline Test Metrics ===")
print(f"  Accuracy:  {accuracy(y_test, y_pred):.4f}")
print(f"  Logloss:   {logloss(y_test, y_proba):.4f}")
print(f"  AUC:       {auc(y_test, y_proba):.4f}")
print(f"  Precision: {precision(y_test, y_pred):.4f}")
print(f"  Recall:    {recall(y_test, y_pred):.4f}")
print(f"  F1:        {f1_score(y_test, y_pred):.4f}")

=== Baseline Test Metrics ===
  Accuracy:  0.7864
  Logloss:   0.7123
  AUC:       0.8283
  Precision: 0.6272
  Recall:    0.4813
  F1:        0.5446


In [13]:
# --- 4c. Confusion matrix ---
tp = int(np.sum((y_pred == 1) & (y_test == 1)))
tn = int(np.sum((y_pred == 0) & (y_test == 0)))
fp = int(np.sum((y_pred == 1) & (y_test == 0)))
fn = int(np.sum((y_pred == 0) & (y_test == 1)))

print("=== Confusion Matrix ===")
print(f"                   Pred No   Pred Yes")
print(f"  Actual No       {tn:6d}     {fp:6d}")
print(f"  Actual Yes      {fn:6d}     {tp:6d}")

=== Confusion Matrix ===
                   Pred No   Pred Yes
  Actual No          928        107
  Actual Yes         194        180


---
## 5. Cross-Validation

In [14]:
# --- 5a. K-Fold CV ---
def k_fold_cv(X, y, n_folds=5, **model_params):
    """Stratified k-fold cross-validation."""
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    np.random.shuffle(pos_idx)
    np.random.shuffle(neg_idx)

    pos_folds = np.array_split(pos_idx, n_folds)
    neg_folds = np.array_split(neg_idx, n_folds)

    fold_metrics = []
    n = len(y)

    for fold in range(n_folds):
        test_idx_fold = np.concatenate([pos_folds[fold], neg_folds[fold]])
        train_mask = np.ones(n, dtype=bool)
        train_mask[test_idx_fold] = False

        X_tr, X_val = X[train_mask], X[test_idx_fold]
        y_tr, y_val = y[train_mask], y[test_idx_fold]

        clf = LGBMClassifier(**model_params)
        t0 = time.time()
        clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)])
        elapsed = time.time() - t0

        yp = clf.predict(X_val)
        ypr = clf.predict_proba(X_val)[:, 1]

        m = {
            "accuracy": accuracy(y_val, yp),
            "logloss": logloss(y_val, ypr),
            "auc": auc(y_val, ypr),
            "f1": f1_score(y_val, yp),
            "precision": precision(y_val, yp),
            "recall": recall(y_val, yp),
            "time": elapsed,
        }
        fold_metrics.append(m)
        print(f"  Fold {fold+1}: acc={m['accuracy']:.4f}  auc={m['auc']:.4f}  "
              f"f1={m['f1']:.4f}  logloss={m['logloss']:.4f}  ({elapsed:.1f}s)")

    return fold_metrics

print("Defined stratified k-fold CV")

Defined stratified k-fold CV


In [15]:
# --- 5b. Run 5-fold CV ---
print("=== 5-Fold Cross-Validation ===")
cv_params = dict(
    objective="binary",
    n_estimators=150,
    learning_rate=0.1,
    num_leaves=31,
    min_data_in_leaf=20,
    lambda_l2=1.0,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    max_bin=63,
    early_stopping_rounds=20,
    verbose=False,
    seed=42,
)

fold_results = k_fold_cv(X_scaled, y, n_folds=5, **cv_params)

=== 5-Fold Cross-Validation ===
  Fold 1: acc=0.7878  auc=0.8342  f1=0.5698  logloss=0.7152  (4.6s)
  Fold 2: acc=0.8027  auc=0.8369  f1=0.5710  logloss=0.7101  (3.9s)
  Fold 3: acc=0.8027  auc=0.8395  f1=0.5762  logloss=0.7099  (6.1s)
  Fold 4: acc=0.7871  auc=0.8325  f1=0.5614  logloss=0.7160  (3.6s)
  Fold 5: acc=0.8067  auc=0.8414  f1=0.5988  logloss=0.7113  (4.0s)


In [16]:
# --- 5c. Aggregate CV results ---
print("=== Cross-Validation Summary ===")
metric_names = ["accuracy", "logloss", "auc", "f1", "precision", "recall"]
print(f"  {'Metric':<12s} {'Mean':>8s} {'Std':>8s} {'Min':>8s} {'Max':>8s}")
print(f"  {'-'*46}")
for m in metric_names:
    vals = [fm[m] for fm in fold_results]
    print(f"  {m:<12s} {np.mean(vals):8.4f} {np.std(vals):8.4f} {np.min(vals):8.4f} {np.max(vals):8.4f}")

times = [fm["time"] for fm in fold_results]
print(f"\n  Total CV time: {sum(times):.1f}s  (avg {np.mean(times):.1f}s per fold)")

=== Cross-Validation Summary ===
  Metric           Mean      Std      Min      Max
  ----------------------------------------------
  accuracy       0.7974   0.0083   0.7871   0.8067
  logloss        0.7125   0.0026   0.7099   0.7160
  auc            0.8369   0.0033   0.8325   0.8414
  f1             0.5754   0.0126   0.5614   0.5988
  precision      0.6494   0.0258   0.6168   0.6752
  recall         0.5174   0.0176   0.4947   0.5442

  Total CV time: 22.2s  (avg 4.4s per fold)


---
## 6. Hyperparameter Tuning

Grid search over key parameters using 3-fold CV AUC.

In [17]:
def grid_search_cv(X, y, param_grid, n_folds=3):
    """Exhaustive grid search with stratified k-fold CV."""
    keys = list(param_grid.keys())
    combos = list(product(*param_grid.values()))

    n = len(y)
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    np.random.shuffle(pos_idx)
    np.random.shuffle(neg_idx)
    pos_folds = np.array_split(pos_idx, n_folds)
    neg_folds = np.array_split(neg_idx, n_folds)

    print(f"Grid search: {len(combos)} combos x {n_folds}-fold CV")
    results = []

    for i, combo in enumerate(combos):
        params = dict(zip(keys, combo))
        params.update({"objective": "binary", "n_estimators": 100, "verbose": False,
                       "early_stopping_rounds": 15, "max_bin": 63, "seed": 42})

        fold_scores = []
        for f in range(n_folds):
            test_f = np.concatenate([pos_folds[f], neg_folds[f]])
            mask = np.ones(n, dtype=bool)
            mask[test_f] = False

            clf = LGBMClassifier(**params)
            clf.fit(X[mask], y[mask], eval_set=[(X[test_f], y[test_f])])
            ypr = clf.predict_proba(X[test_f])[:, 1]
            fold_scores.append(auc(y[test_f], ypr))

        mean_s = np.mean(fold_scores)
        std_s = np.std(fold_scores)
        results.append({"params": params, "score": mean_s, "std": std_s})
        short = {k: v for k, v in params.items() if k in keys}
        print(f"  [{i+1:2d}/{len(combos)}] {short}  ->  AUC={mean_s:.4f} (+/-{std_s:.4f})")

    results.sort(key=lambda x: x["score"], reverse=True)
    return results

In [18]:
param_grid = {
    "num_leaves":        [15, 31, 63],
    "learning_rate":     [0.05, 0.1],
    "lambda_l2":         [0.5, 1.0, 5.0],
    "min_data_in_leaf":  [10, 20],
}

print("=== Grid Search ===")
t0 = time.time()
search_results = grid_search_cv(X_scaled, y, param_grid, n_folds=3)
print(f"\nGrid search done in {time.time()-t0:.1f}s")

print("\n=== Top 5 Configurations ===")
for i, r in enumerate(search_results[:5]):
    p = r["params"]
    print(f"  #{i+1}: AUC={r['score']:.4f} (+/-{r['std']:.4f})  "
          f"lr={p['learning_rate']} leaves={p['num_leaves']} l2={p['lambda_l2']} min_leaf={p['min_data_in_leaf']}")

=== Grid Search ===
Grid search: 36 combos x 3-fold CV
  [ 1/36] {'num_leaves': 15, 'learning_rate': 0.05, 'lambda_l2': 0.5, 'min_data_in_leaf': 10}  ->  AUC=0.8402 (+/-0.0038)
  [ 2/36] {'num_leaves': 15, 'learning_rate': 0.05, 'lambda_l2': 0.5, 'min_data_in_leaf': 20}  ->  AUC=0.8401 (+/-0.0040)
  [ 3/36] {'num_leaves': 15, 'learning_rate': 0.05, 'lambda_l2': 1.0, 'min_data_in_leaf': 10}  ->  AUC=0.8406 (+/-0.0040)
  [ 4/36] {'num_leaves': 15, 'learning_rate': 0.05, 'lambda_l2': 1.0, 'min_data_in_leaf': 20}  ->  AUC=0.8400 (+/-0.0041)
  [ 5/36] {'num_leaves': 15, 'learning_rate': 0.05, 'lambda_l2': 5.0, 'min_data_in_leaf': 10}  ->  AUC=0.8415 (+/-0.0039)
  [ 6/36] {'num_leaves': 15, 'learning_rate': 0.05, 'lambda_l2': 5.0, 'min_data_in_leaf': 20}  ->  AUC=0.8411 (+/-0.0034)
  [ 7/36] {'num_leaves': 15, 'learning_rate': 0.1, 'lambda_l2': 0.5, 'min_data_in_leaf': 10}  ->  AUC=0.8392 (+/-0.0037)
  [ 8/36] {'num_leaves': 15, 'learning_rate': 0.1, 'lambda_l2': 0.5, 'min_data_in_leaf': 20}

---
## 7. Final Model

Retrain with the best parameters on the full train set, evaluate on the held-out test set.

In [19]:
# --- 7a. Train final model ---
best = search_results[0]["params"]
print(f"Best params: leaves={best['num_leaves']}, lr={best['learning_rate']}, "
      f"l2={best['lambda_l2']}, min_leaf={best['min_data_in_leaf']}")

final_model = LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=best["learning_rate"],
    num_leaves=best["num_leaves"],
    min_data_in_leaf=best["min_data_in_leaf"],
    lambda_l2=best["lambda_l2"],
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    max_bin=63,
    early_stopping_rounds=25,
    verbose=True,
    seed=42,
)

t0 = time.time()
final_model.fit(X_train, y_train, eval_set=[(X_test, y_test)])
print(f"\nTraining time: {time.time()-t0:.1f}s")

Best params: leaves=15, lr=0.05, l2=5.0, min_leaf=10
[    0] train loss: 0.564829 | valid_0: 0.566114
[    1] train loss: 0.553239 | valid_0: 0.556305
[    2] train loss: 0.541975 | valid_0: 0.545981
[    3] train loss: 0.532402 | valid_0: 0.537212
[    4] train loss: 0.523512 | valid_0: 0.529286
[    5] train loss: 0.514790 | valid_0: 0.521780
[    6] train loss: 0.507742 | valid_0: 0.515810
[    7] train loss: 0.501021 | valid_0: 0.510273
[    8] train loss: 0.494144 | valid_0: 0.504154
[    9] train loss: 0.487835 | valid_0: 0.498786
[   10] train loss: 0.482093 | valid_0: 0.493723
[   11] train loss: 0.477081 | valid_0: 0.489547
[   12] train loss: 0.472530 | valid_0: 0.485251
[   13] train loss: 0.467773 | valid_0: 0.481258
[   14] train loss: 0.463428 | valid_0: 0.477542
[   15] train loss: 0.459390 | valid_0: 0.474115
[   16] train loss: 0.455584 | valid_0: 0.470999
[   17] train loss: 0.452112 | valid_0: 0.467682
[   18] train loss: 0.448832 | valid_0: 0.464894
[   19] train lo

In [20]:
# --- 7b. Final evaluation ---
y_pred_final = final_model.predict(X_test)
y_proba_final = final_model.predict_proba(X_test)[:, 1]

print("=== Final Model — Test Set Metrics ===")
print(f"  Accuracy:  {accuracy(y_test, y_pred_final):.4f}")
print(f"  Logloss:   {logloss(y_test, y_proba_final):.4f}")
print(f"  AUC:       {auc(y_test, y_proba_final):.4f}")
print(f"  Precision: {precision(y_test, y_pred_final):.4f}")
print(f"  Recall:    {recall(y_test, y_pred_final):.4f}")
print(f"  F1:        {f1_score(y_test, y_pred_final):.4f}")

tp = int(np.sum((y_pred_final == 1) & (y_test == 1)))
tn = int(np.sum((y_pred_final == 0) & (y_test == 0)))
fp = int(np.sum((y_pred_final == 1) & (y_test == 0)))
fn = int(np.sum((y_pred_final == 0) & (y_test == 1)))
print(f"\n=== Confusion Matrix ===")
print(f"                   Pred No   Pred Yes")
print(f"  Actual No       {tn:6d}     {fp:6d}")
print(f"  Actual Yes      {fn:6d}     {tp:6d}")

=== Final Model — Test Set Metrics ===
  Accuracy:  0.7871
  Logloss:   0.7135
  AUC:       0.8326
  Precision: 0.6267
  Recall:    0.4893
  F1:        0.5495

=== Confusion Matrix ===
                   Pred No   Pred Yes
  Actual No          926        109
  Actual Yes         191        183


In [21]:
# --- 7c. Save model ---
model_path = os.path.join(PROJECT_ROOT, "churn_model.json")
final_model.save_model(model_path)
print(f"Model saved to {model_path}")

Model saved to /home/ei/Projects/UNI/LightGBMLibrary/churn_model.json


---
## 8. Summary

In [22]:
cv_auc_mean = search_results[0]["score"]
cv_auc_std = search_results[0]["std"]

print("=" * 60)
print("  SUMMARY")
print("=" * 60)
print(f"  Dataset:          Telco Customer Churn")
print(f"  Samples:          {len(y)}  (train={len(y_train)}, test={len(y_test)})")
print(f"  Features:         {X.shape[1]}  ({len(numeric_cols)} numeric, {len(categorical_cols)} categorical)")
print(f"  Target:           Churn (Yes/No, rate={y.mean():.1%})")
print(f"  Best params:      leaves={best['num_leaves']}, lr={best['learning_rate']}, "
      f"l2={best['lambda_l2']}, min_leaf={best['min_data_in_leaf']}")
print(f"  CV AUC (best):    {cv_auc_mean:.4f} (+/-{cv_auc_std:.4f})")
print(f"  Test AUC:         {auc(y_test, y_proba_final):.4f}")
print(f"  Test F1:          {f1_score(y_test, y_pred_final):.4f}")
print(f"  Test Accuracy:    {accuracy(y_test, y_pred_final):.4f}")
print("=" * 60)

  SUMMARY
  Dataset:          Telco Customer Churn
  Samples:          7043  (train=5634, test=1409)
  Features:         19  (4 numeric, 15 categorical)
  Target:           Churn (Yes/No, rate=26.5%)
  Best params:      leaves=15, lr=0.05, l2=5.0, min_leaf=10
  CV AUC (best):    0.8415 (+/-0.0039)
  Test AUC:         0.8326
  Test F1:          0.5495
  Test Accuracy:    0.7871
